In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")


**Task 1: Getting Started with AstraDB (DataStax)**
1. Create a DataStax AstraDB account.
2. Create a vector database.
3. Generate application token & DB endpoint.


In [6]:
if os.getenv("ASTRA_DB_API_ENDPOINT") is None:
    raise ValueError("ASTRA_DB_API_ENDPOINT is not set")
if os.getenv("ASTRA_DB_APPLICATION_TOKEN") is None:
    raise ValueError("ASTRA_DB_APPLICATION_TOKEN is not set")


**Task 2: Connect LangChain with AstraDB**
1. Install AstraDB LangChain integration.
2. Connect to AstraDB using:
   - Token
   - Endpoint
3. Verify connection.

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_astradb import AstraDBVectorStore

/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/astrapy/admin/admin.py:67: UserWarning: SSL connection reuse disabled due to a Python 3.12.[0-11] bug. This may reduce performance under certain workloads. Please upgrade to Python 3.12.12 or newer if possible.
  from astrapy.utils.api_commander import APICommander
/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [5]:
vstore = AstraDBVectorStore(
    embedding = embeddings,
    collection_name = 'pdf_rag',
    api_endpoint = os.environ['ASTRA_DB_API_ENDPOINT'],
    token = os.environ['ASTRA_DB_APPLICATION_TOKEN'],
)
print('AstraDB vector store ready')

AstraDB vector store ready


**Task 3: Load & Split PDF Document**
1. Load a PDF using LangChain PDF loader.
2. Split text into chunks.
3. Prepare documents for embedding.


In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_51063/2300941456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
loader = PyPDFLoader("hp1.pdf")
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)

In [9]:
data = loader.load()

In [10]:
chunks = splitter.split_documents(data)
print(f"Splitted into {len(chunks)} chunks")
print(f"sample chunk: {chunks[0]}")

Splitted into 455 chunks
sample chunk: page_content='/ 
THE BOY WHO LIVED 
Mr. and Mrs. Dursley, of number four, Privet Drive, 
were proud to say that they were perfectly normal, 
thank you very much. They were the last people youâ€™d 
expect to be involved in anything strange or 
mysterious, because they just didnâ€™t hold with such 
nonsense. 
Mr. Dursley was the director of a firm called 
Grunnings, which made drills. He was a big, beefy 
man with hardly any neck, although he did have a 
very large mustache. Mrs. Dursley was thin and 
blonde and had nearly twice the usual amount of 
neck, which came in very useful as she spent so 
much of her time craning over garden fences, spying 
on the neighbors. The Dursley s had a small son 
called Dudley and in their opinion there was no finer 
boy anywhere. 
The Dursleys had everything they wanted, but they 
also had a secret, and their greatest fear was that 
somebody would discover it. They didnâ€™t think they 
could bear it if anyone foun

**Task 4: Store Embeddings in AstraDB**
1. Use embedding model (HuggingFace / OpenAI).
2. Store document embeddings in AstraDB vector store.
3. Verify data persistence.


In [23]:
# wipe duplicates from earlier runs, then load once
# (delete() needs ids — clear() empties the whole collection)
vstore.clear()
print("collection cleared")


collection cleared


In [24]:

ids = vstore.add_documents(chunks)
print(f"stored {len(ids)} chunks once")


stored 455 chunks once


In [25]:
hits = vstore.similarity_search("Who is Harry Potter?", k=2)
print(f"similarity_search returned {len(hits)} docs")
print(hits[0].page_content[:300])

similarity_search returned 2 docs
never even heard of Hogwarts until they get the letter, 
imagine. I think they should keep it in the old 
wizarding families. Whatâ€™s your surname, anyway?â€   
But before Harry could answer, Madam Malkin said, 
â€œThatâ€™s you done, my dear,â€   and Harry, not sorry for 
an excuse to stop talkin


**Task 5: PDF Query RAG Application**
Build a RAG pipeline:
PDF → Splitter → Embeddings → AstraDB → Retriever → LLM → Answer
1. Accept user questions.
2. Retrieve relevant PDF chunks.
3. Generate grounded answers.

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [28]:
retriever = vstore.as_retriever(search_kwargs={"k": 10})


In [29]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You answer questions about the Harry Potter PDF using ONLY the context below. "
        "Quote or paraphrase facts from the context. "
        "If the context does not contain the answer, reply exactly: I don't know.\n\n"
        "Context:\n{context}",
    ),
    ("human", "{question}"),
])


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


In [30]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)









**Task 6: Testing & Validation**
1. Ask at least 5 questions from the PDF.
2. Verify answers come from document content.
3. Handle out-of-context questions gracefully.


In [31]:
# Task 6 — 5+ in-doc questions + one out-of-context
questions = [
    "Where do the Dursleys live?",
    "Who left baby Harry on the Dursleys' doorstep?",
    "What is Hogwarts?",
    "Who tells Harry he is a wizard?",
    "What is the name of Hagrid's dog?",
    "What is the capital of France?",  # not in PDF → don't know
]

for q in questions:
    print("=" * 60)
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print()


Q: Where do the Dursleys live?
A: The Dursleys live at number four, Privet Drive.

Q: Who left baby Harry on the Dursleys' doorstep?
A: Dumbledore left baby Harry on the Dursleys' doorstep. He took Harry in his arms and turned toward the Dursleys' house.

Q: What is Hogwarts?
A: Hogwarts is a school of witchcraft and wizardry where students are sorted into four Houses: Gryffindor, Hufflepuff, Ravenclaw, and Slytherin. Each House has its own noble history and produces outstanding witches and wizards. While at Hogwarts, students earn House points for their triumphs and lose points for rule-breaking, with the House that has the most points at the end of the year being awarded the House cup.

Q: Who tells Harry he is a wizard?
A: Hagrid tells Harry he is a wizard.

Q: What is the name of Hagrid's dog?
A: Hagrid's dog is named Fang.

Q: What is the capital of France?
A: I don't know.

